In [0]:
%run ../common/config

In [0]:
from pyspark.sql.functions import *

In [0]:
patients_df= spark.read.table(f"{env_catalog}.bronze.patients")

In [0]:
silver_patients = (
    patients_df
    .dropDuplicates(["Id"])
    .withColumn(
        "birth_date",
        to_date(col("BIRTHDATE"))
    )
    .withColumn(
        "death_date",
        to_date(col("DEATHDATE"))
    )
    .select(
        col("Id").alias("patient_id"),
        "birth_date",
        "death_date",
        col("FIRST").alias("first_name"),
        col("LAST").alias("last_name"),
        col("GENDER").alias("gender"),
        col("RACE").alias("race"),
        col("ETHNICITY").alias("ethnicity"),
        col("CITY").alias("city"),
        col("STATE").alias("state"),
        col("ZIP").alias("zip_code"),
        col("HEALTHCARE_EXPENSES").alias("healthcare_expenses"),
        col("HEALTHCARE_COVERAGE").alias("healthcare_coverage"),
        col("INCOME").alias("income")
    )
    .withColumn(
        "silver_load_timestamp",
        current_timestamp()
    )
)

In [0]:
invalid_patients = silver_patients.filter(
    col("patient_id").isNull()
)

print(invalid_patients.count())

In [0]:
(
    silver_patients.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{catalog}.silver.patients"
    )
)